In [1]:
from pathlib import Path
import fitz, os, re, json, textwrap, pandas as pd
pdf_path=Path('/workspace/jacobson-explosibility-of-metal-powders-1964.pdf')
doc=fitz.open(pdf_path)
print('pages', len(doc), 'size', pdf_path.stat().st_size)
for p in [7,8,9,10,12,17,21,26,27,28,29,30,31]:
    txt=doc[p].get_text('text')
    print(p+1, len(txt), repr(txt[:120]))

pages 35 size 1375259
8 1896 'EXPLOSIBILITY OF METAL\nPOWDERS\n1\nby\nMurray Jacobson ,\nAustin R. Cooper , 2 and John Nagy3\nABSTRACT\nData obtained in the '
9 2707 '2\ndata on metals were published.6\nThe equipment and procedures used in dust\nexplosibility evaluation have been described'
10 2693 '3\nThe relative hazard of a dust\nis also given by adjective ratings of none ,\nfire , weak , moderate ,\nstrong ,\nand sever'
11 2945 '4\nTABLE 1 .\n-\nIgnition\nLine\ntemperature , 1\nMaterial\nNo.\nC\nCloud\nLayer\nIgnition and explosibility of metal powders\nMinim'
13 1834 '6\nMAXIMUM\nRATE\nOF\nPRESSURE\nRISE\n,psi\n/\nsec\nMAXIMUM\nPRESSURE\n,psig\n130\n110\n90\n70\n50\n20,000\n15,000\n10,000\n5,000\n0\n0.5\nFlak'
18 4838 'APPENDIX\n11\nTABLE A- 1 . ·\nExplo-\nLine Sample sibility\nMaterial\nindex\nDescription , explosibility index , and particle s'
22 2465 'Minimum igniting\nIgnition\nLine\nSample\nsensitivity\nenergy for\ndust , millijoules\nCloud\nLayer\nCloud\nLayer\nTABLE A- 2 . 

In [2]:
import os
from e14c.edison_client import get_edison_client
from e14c.subagents import submit_task
client=get_edison_client()
print({k: bool(os.environ.get(k)) for k in ['PROJECT_ID','PERSONA_ID']})
# A sub-agent project cannot be created in this runtime because PERSONA_ID is unavailable.
# Continue the extraction locally rather than weakening provenance requirements.
project_id=os.environ.get('PROJECT_ID')
task_id=None
if project_id:
    query='''Find and retrieve the best available machine-readable or higher-resolution public copy of U.S. Bureau of Mines Report of Investigations 6516 (1964), Explosibility of Metal Powders by Jacobson, Cooper, and Nagy. Focus on sources that can resolve OCR ambiguities in all tables, especially appendix Tables A-1, A-2, and A-3 and missing scan page 21. Save retrieved files under downloads/. Report exact source URLs and file paths. Do not invent unreadable values.'''
    task_id=await submit_task(query, project_id=project_id, agent_type='data-retrieval')
print('project_id',project_id,'task_id',task_id)

{'PROJECT_ID': False, 'PERSONA_ID': False}
project_id None task_id None


In [3]:
import subprocess, sys
subprocess.run([sys.executable,'-m','pip','install','pdfplumber','-q'],check=True)
import pdfplumber, pandas as pd, re, json, os, textwrap, numpy as np
with pdfplumber.open(pdf_path) as pdf:
    for pno in [11,18,22,27,28,29,30,31,32]:
        page=pdf.pages[pno-1]
        tables=page.extract_tables({'vertical_strategy':'text','horizontal_strategy':'text','intersection_tolerance':5,'snap_tolerance':3,'join_tolerance':3,'text_tolerance':2})
        print('page',pno,'tables',len(tables),[ (len(t), max(map(len,t)) if t else 0) for t in tables])
        if tables: print(tables[0][:3])

page 11 tables 1 [(73, 9)]
[['4', '', '', '', '', '', '', '', ''], ['', '', '', '', '', '', '', '', ''], ['TAB', 'LE 1. - Ign', 'ition', 'and expl', 'osibility of', 'metal powde', 'rs', '', '']]


page 18 tables 1 [(78, 16)]
[['', '', '', '', 'TABLE A-1. · D', 'escriptio', 'n, explosibility', 'index,', 'and parti', 'cle size of', 'dusts', '', '', '', '', ''], ['', '', '', '', '', '', '', '', '', '', '', '', '', '', '', ''], ['', 'Exp', 'lo-', '', '', '', '', '', '', '', 'Part', 'icle', 'size, p', 'ercen', 't Sub', 'sie']]


page 22 tables 1 [(79, 12)]
[['', '', 'TABLE A-2', '. - Igni', 'tion s', 'ensiti', 'vity, relative fla', 'mmability, and', 'limiting oxygen', '', '', ''], ['', '', '', '', '', 'con', 'centration of atmo', 'sphere for dus', 'ts', '', '', ''], ['', '', '', '', '', '', '', '', '', '', '', '']]


page 27 tables 1 [(73, 21)]
[['', '', '', 'MAavxeirmaugme', '', '', '', '19,0', '0106,80,00400,057,00500,00003,', '300', '', '', '', '', '3,', '4002,060,', '405,08050', '0', '', '', '2'], ['', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', ',', '000', '', '', ',200'], ['', '', ',rppirrseeess', 's-ure', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '']]


page 28 tables 1 [(79, 22)]
[['', '', '', '', '0\n0', '0\n0', '', '0\n00000', '', '0', '0', '', '', '', '', '', '', '', '', '', '', ''], ['0\n0', '', '', '', '0\n,\n8', '0\n,\n3', '', '00000\n070,9\n,,,0,', '', '0\n5\n,', '0\n2\n,', '', '', '', '', '', '', '', '', '', '', '0\n0'], ['1', '', '', '', '1', '1', '', '82911', '', '3 :', '3', '', '', '', '', '', '', '', '', '', '', '8']]


page 29 tables 1 [(65, 23)]
[['', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', ''], ['22', '', 'Maximum', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', ''], ['', '', 'Average', '', '', '30020', '0', '', '', '', '', '5', '00 40', '0', '', '', '', '', '', '', '', '', '']]


page 30 tables 1 [(74, 22)]
[['', '', '', '', '', '', '', '0\n0', '', '0\n0', '0\n0', '0 0\n0', '0\n0', '00\n00', '0\n0', '0\n0', '0 00\n0', '', '', '0', '', ''], ['', '', '', '', '', '', '', '6', '', '02\n0', '6', '0\n0 9', '0', '280\n5', '5', ',', '5 4', '', '', '0\n0', '', ''], ['', '', '', '', '', '', '', ',\n2', '', ',\n81', ',\n2', ', ,\n3 3', ',\n6', ',,\n444', ',\n5', '2\n1 :', ', ,2\n5', '', '', ',\n6', '', '']]


page 31 tables 1 [(82, 21)]
[['', 'MAav', 'xeirmaugme\n5,000', '', '', '15', '0', '', '', '', '100', '', '100 30', '0', '', '', '', '', '', '', ''], ['', '', '9,000', '', '1,', '400', '', '7,45,0900', '0', '8,50', '0 2,39,0', '8000', '', '4,', '700', '', '5,', '050,505,0450,0', '208,0220,0500', '0 3', ',1'], [',rpir', 'seessu', 're', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '']]
page 32 tables 1 [(68, 21)]
[['', '', '', '0', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', ''], ['', '', '', '00\n00\n3,', '0', '004', '0\n0\n80', '004', '0\n0\n7', '', '0\n0\n3', '', '', '', '', '', '', '', '00', '', ''], ['', '', '', ',1\n41', '0\n3', ',1', ',0\n34', ',1', ',\n1', '', ',\n1', '', '', '', '', '', '', '', '00\n12', '', '']]


In [4]:
from e14c.web import web_search
results=await web_search('"Explosibility of Metal Powders" RI 6516 PDF OCR',num_results=10)
for i,r in enumerate(results):
    print(i,r['title'],r['url'])

0 OneMine | RI 6516 Explosibility of Metal Powders https://www.onemine.org/documents/ri-6516-explosibility-of-metal-powders
1 UNCLASSIFIED https://www.govinfo.gov/content/pkg/GOVPUB-I-PURL-gpo88221/pdf/GOVPUB-I-PURL-gpo88221.pdf
2 TRAIL - Technical Report Archive and Image Library - Detail https://www.technicalreports.org/trail/detail/21562/
3 (PDF) Explosibility of Metal Powders https://www.academia.edu/145221005/Explosibility_of_Metal_Powders
4 Explosibility of Nano-Sized Metal Powders https://stacks.cdc.gov/view/cdc/195110/cdc_195110_DS1.pdf
5  https://hal.univ-lorraine.fr/hal-02372373v1/document
6 OAR Box 1955 2) ll-D-250 https://downloads.regulations.gov/EPA-HQ-OAR-2002-0014-0330/content.pdf
7 INFLAMMABILITY AND EXPLOSIBILITY OF METAL POWDERS (Technical Report) | OSTI.GOV https://www.osti.gov/biblio/4319179
8 ^x- ^ rs^ https://sanonofresafety.org/wp-content/uploads/2014/12/4410914explosivezirconiumdivofmines.pdf
9 ), the maximum rate of pressure rise (( ⁄) ) https://irinsubria.uni

In [5]:
from e14c.web import fetch_parsed_page
trail=await fetch_parsed_page('https://www.technicalreports.org/trail/detail/21562/')
print(trail[:5000])

TRAIL - Technical Report Archive and Image Library - Detail

[skip to content](https://www.technicalreports.org/trail/detail/21562/#results)

# 

|  |  |
|  |  |

# Explosibility of metal powders

| Author:             | Jacobson, M.         |
| ------------------- | -------------------- |
| Additional Authors: | Nagy, John.,   Coope |
| Year:               | 1964                 |
| Document Type:      | BMRA                 |
| Issuing Agency:     | U.S. Dept. of the In |
| SUDOC:              | I 28.23:6516         |
| Series:             | Report of investigat |
| Report Number:      | 6516                 |
| Subjects:           | Dust explosions      |
|                     | Metal powders        |
| Link:               | https://hdl.handle.n |

[< New search](https://www.technicalreports.org/trail/search/)

[![The Center for Research Libraries](https://www.technicalreports.org/trail-static/graphics/crl_logo_full.svg "width=350")](http://www.crl.edu/)

* [About TRAIL](https://www

In [6]:
# Inspect search result metadata and links for direct archive handles.
r=results[2]
print(json.dumps({k:r.get(k) for k in ['title','url','snippet','links']},indent=2)[:12000])
print('\nOneMine links:')
print(json.dumps(results[0].get('links'),indent=2)[:8000])

{
  "title": "TRAIL - Technical Report Archive and Image Library - Detail",
  "url": "https://www.technicalreports.org/trail/detail/21562/",
  "snippet": "# Explosibility of metal powders\n...\n| Author: | Jacobson, M. |\n| --- | --- |\n| Additional Authors: | Nagy, John., Cooper, Austin R. |\n| Year: | 1964 |\n| Document Type: | BMRA |\n| Issuing Agency: | U.S. Dept. of the Interior, Bureau of Mines, |\n| SUDOC: | I 28.23:6516 |\n| Series: | Report of investigations / United States Department of the Interior, Bureau of Mines ; |\n| Report Number: | 6516 |\n| Subjects: | Dust explosions |\n| Metal powders |\n| Link: | https://hdl.handle.net/2027/mdp.39015078519538 |",
  "links": null
}

OneMine links:
null


In [7]:
from e14c.web import fetch_parsed_page
handle=await fetch_parsed_page('https://hdl.handle.net/2027/mdp.39015078519538')
print(handle[:8000])

\#3 - Explosibility of metal powders / by Murray Jacobson, Austin ... - Full View | HathiTrust Digital Library 

# HathiTrust cookie settings

* [Skip to page content](https://babel.hathitrust.org/cgi/pt?id=mdp.39015078519538&seq=3#main)
* [Skip to text-only view](https://babel.hathitrust.org/cgi/ssd?id=mdp.39015078519538)
* [Skip to search in this text](https://babel.hathitrust.org/cgi/pt?id=mdp.39015078519538&seq=3#input-search-text)

# Questions?

* + [Welcome to HathiTrust](https://www.hathitrust.org/about)
    + [Our Mission & History](https://www.hathitrust.org/about/mission-history)
    + [HathiTrust Research Center (HTRC)](https://www.hathitrust.org/about/research-center)
    + [Governance](https://www.hathitrust.org/about/governance)
    + [Our Team](https://www.hathitrust.org/about/our-team)
* + [About the Collection](https://www.hathitrust.org/the-collection)
    + [Featured Collections](https://babel.hathitrust.org/cgi/mb?a=listcs&colltype=featured)
    + [How to Search & A

In [8]:
import requests, re, os, json, textwrap
urls={
'pdf':'https://babel.hathitrust.org/cgi/imgsrv/download/pdf?id=mdp.39015078519538&attachment=1&tracker=D1&seq=3',
'ocr':'https://babel.hathitrust.org/cgi/ssd?id=mdp.39015078519538&seq=3'
}
for k,u in urls.items():
 r=requests.get(u,timeout=60,headers={'User-Agent':'Mozilla/5.0'})
 print(k,r.status_code,r.headers.get('content-type'),len(r.content),r.url)
 Path(f'/workspace/hathi_{k}.bin').write_bytes(r.content)

pdf 403 text/html; charset=UTF-8 66401 https://babel.hathitrust.org/cgi/imgsrv/download/pdf?id=mdp.39015078519538&attachment=1&tracker=D1&seq=3
ocr 403 text/html; charset=UTF-8 5610 https://babel.hathitrust.org/cgi/ssd?id=mdp.39015078519538&seq=3


In [9]:
# Build a layout-preserving text export directly from PDF word coordinates.
from pathlib import Path
import fitz, statistics, math

def page_to_layout(page, width=180):
    words=page.get_text('words', sort=True)  # x0,y0,x1,y1,text,block,line,word
    if not words: return ''
    x_min=min(w[0] for w in words); x_max=max(w[2] for w in words)
    scale=(width-1)/max(1,(x_max-x_min))
    # group by PDF line identity where possible
    lines={}
    for w in words:
        key=(w[5],w[6])
        lines.setdefault(key,[]).append(w)
    ordered=sorted(lines.values(), key=lambda ws:(min(w[1] for w in ws),min(w[0] for w in ws)))
    out=[]
    for ws in ordered:
        ws=sorted(ws,key=lambda w:w[0]); chars=[' ']*width
        for w in ws:
            pos=max(0,min(width-1,round((w[0]-x_min)*scale)))
            s=w[4]
            for j,ch in enumerate(s):
                if pos+j<width: chars[pos+j]=ch
        out.append(''.join(chars).rstrip())
    return '\n'.join(out)

pages_layout=[page_to_layout(doc[i]) for i in range(len(doc))]
layout_path=Path('/workspace/ri6516_layout.txt')
layout_path.write_text('\n\f\n'.join(pages_layout))
print('pages',len(pages_layout),'chars',sum(map(len,pages_layout)))
print(pages_layout[27][:6000])

pages 35 chars 810862
                                                    13,000
                                                                      10,000
                                        18,000
                                                                    9,000
                                                                8,000
                                                                                   3,500
                                                                                            3,200
                                                                 2,700
                                                                       1,900
                                                                                                                                                                                21
                                                                                                                                         

In [10]:
# Render the hardest table pages for direct visual verification.
outdir=Path('/workspace/ri6516_pages'); outdir.mkdir(exist_ok=True)
for pno in [11,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32]:
    page=doc[pno-1]
    pix=page.get_pixmap(matrix=fitz.Matrix(3,3),alpha=False)
    pix.save(outdir/f'page_{pno:02d}.png')
print('rendered',len(list(outdir.glob('*.png'))),'pages to',outdir)
print((outdir/'page_28.png').stat().st_size)

rendered 16 pages to /workspace/ri6516_pages
610070


In [11]:
# Test PyMuPDF's ruled-table detector, which can preserve column boundaries more reliably than OCR text flow.
for pno in [11,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32]:
    tabs=doc[pno-1].find_tables().tables
    print('page',pno,'tables',len(tabs),[(t.row_count,t.col_count) for t in tabs])

Consider using the pymupdf_layout package for a greatly improved page layout analysis.
page 11 tables 0 []
page 17 tables 0 []


page 18 tables 0 []
page 19 tables 0 []
page 20 tables 0 []


page 21 tables 0 []
page 22 tables 0 []
page 23 tables 0 []
page 24 tables 0 []
page 25 tables 0 []


page 26 tables 0 []
page 27 tables 0 []
page 28 tables 0 []
page 29 tables 0 []
page 30 tables 0 []


page 31 tables 0 []
page 32 tables 0 []


In [12]:
import subprocess, sys
subprocess.run([sys.executable,'-m','pip','install','pymupdf4llm','-q'],check=True)
import pymupdf4llm
md_auto=pymupdf4llm.to_markdown(str(pdf_path), page_chunks=False, write_images=False, table_strategy='lines_strict')
Path('/workspace/ri6516_auto.md').write_text(md_auto)
print('chars',len(md_auto),'tables',md_auto.count('|---'),'lines',md_auto.count('\n'))
print(md_auto[md_auto.find('TABLE 1'):md_auto.find('TABLE 1')+2500])

chars 90060 tables 118 lines 1055
TABLE 1 . - Ignition and explosibility of metal powders 

|Material|Line<br>No.|Igni<br>temper<br>|tion<br>ature,1<br>C<br>|Minimum<br>explosive<br>concen-<br>|Minimum¹<br>igniting<br>energy for<br>|Maximum<br>pressure,<br>|Maximum<br>rate of<br>pressure<br>|Index of<br>explo-<br>|
|---|---|---|---|---|---|---|---|---|
|||Cloud|Layer|tration,<br>|dust cloud,<br>|psig|rise,<br>|sibility|
||||Sev|oz/cu ft<br>ere|millijoules||psi/sec||
|Aluminum, atomized.....<br>|1<br>|650<br>|760<br>|0.045<br>|50<br>|73<br>|20,000+<br>|>10<br>|
|Aluminum-magnesium alloy.....|249<br>|430<br>|480<br>|.020<br>|80<br>|86<br>|10,000<br>|>10<br>|
|Magnesium.....<br>|159<br>|620<br>|490<br>|.040<br>|40<br>|90<br>|9,000<br>|>10<br>|
|Thorium hydride....<br>Zirconium.....|191<br>228|260<br>20|20<br>190|.080<br>.045|3<br>215|60<br>55|6,500<br>6,500|>10<br>>10|
|Uraniumhydride...|212|20|20|.060|25|43|6,500|>10|
|<br>Titanium....<br>Uranium...|194<br>211|330<br>20|510<br>2100|.045<

In [13]:
# Audit the automated extraction for coverage of all required sections, tables, figures, and appendix line numbers.
md=Path('/workspace/ri6516_auto.md').read_text()
checks={
 'abstract':'ABSTRACT' in md,
 'sample_section':'SAMPLE DESCRIPTION AND REPORTING PROCEDURES' in md,
 'prevention':'PREVENTION OF IGNITION AND EXPLOSION' in md,
 'appendix':'APPENDIX' in md,
 'fig1':'FIGURE 1' in md,
 'fig2':'FIGURE 2' in md,
 'fig3':'FIGURE 3' in md,
 'table1':'TABLE 1' in md,
 'table2':'TABLE 2' in md,
 'table3':'TABLE 3' in md,
 'table4':'TABLE 4' in md,
 'A1':'TABLE A-1' in md,
 'A2':'TABLE A-2' in md,
 'A3':'TABLE A-3' in md,
}
print(checks)
for needle in ['TABLE A-1','TABLE A-2','TABLE A-3','FIGURE 1','FIGURE 2','FIGURE 3']:
    print(needle,[m.start() for m in re.finditer(re.escape(needle),md)][:20])
# show missing-page A3 segment around lines 50-110
pos=md.find('50<br>')
print(md[pos:pos+9000][:9000])

{'abstract': True, 'sample_section': True, 'prevention': True, 'appendix': True, 'fig1': True, 'fig2': True, 'fig3': True, 'table1': True, 'table2': True, 'table3': False, 'table4': True, 'A1': True, 'A2': False, 'A3': False}
TABLE A-1 [35490]
TABLE A-2 []
TABLE A-3 []
FIGURE 1 [19035]
FIGURE 2 [21982]
FIGURE 3 [22451]
50<br>|760<br>|0.045<br>|50<br>|73<br>|20,000+<br>|>10<br>|
|Aluminum-magnesium alloy.....|249<br>|430<br>|480<br>|.020<br>|80<br>|86<br>|10,000<br>|>10<br>|
|Magnesium.....<br>|159<br>|620<br>|490<br>|.040<br>|40<br>|90<br>|9,000<br>|>10<br>|
|Thorium hydride....<br>Zirconium.....|191<br>228|260<br>20|20<br>190|.080<br>.045|3<br>215|60<br>55|6,500<br>6,500|>10<br>>10|
|Uraniumhydride...|212|20|20|.060|25|43|6,500|>10|
|<br>Titanium....<br>Uranium...|194<br>211|330<br>20|510<br>2100|.045<br>.060|25<br>245|70<br>53|5,500<br>3,400|>10<br>>10|
|Thorium..|190|270|280|.075|5|48|3,300|>10|
||||Str|ong|||||
|Titanium hydride..<br>|205<br>|480<br>|540<br>|0.070<br>|60<br>|96<br>

In [14]:
# Compare layout-aware extraction strategies on the most difficult appendix pages.
strategies=['lines','lines_strict','text']
for strat in strategies:
    try:
        out=pymupdf4llm.to_markdown(str(pdf_path),pages=[27],page_chunks=False,write_images=False,table_strategy=strat)
        Path(f'/workspace/page28_{strat}.md').write_text(out)
        print(strat,len(out),out.count('|---'),repr(out[:500]))
    except Exception as e:
        print(strat,'ERROR',e)

lines 44 0 '21 \n\n888 1 1 2233343 19 I 3 1 335 1 eeeee \n\n'


lines_strict 44 0 '21 \n\n888 1 1 2233343 19 I 3 1 335 1 eeeee \n\n'


text 44 0 '21 \n\n888 1 1 2233343 19 I 3 1 335 1 eeeee \n\n'


In [15]:
p=doc[27]
print('page rect',p.rect,'rotation',p.rotation)
d=p.get_text('dict')
for b in d['blocks'][:8]:
    if 'lines' in b:
        for ln in b['lines'][:3]:
            print('dir',ln.get('dir'),'bbox',ln['bbox'],'text',' '.join(s['text'] for s in ln['spans'])[:120])
        print('---')

page rect Rect(0.0, 0.0, 542.3399658203125, 755.2799682617188) rotation 0
dir (0.0, -1.0) bbox (34.0, 681.4999389648438, 38.007999420166016, 689.7799682617188) text 50
dir (0.0, -1.0) bbox (34.20000076293945, 635.3799438476562, 38.20800018310547, 656.0799560546875) text 11101
dir (0.0, -1.0) bbox (34.20000076293945, 612.1959228515625, 38.20800018310547, 619.5799560546875) text .1
---
dir (0.9999500513076782, -0.009999499656260014) bbox (36.15999984741211, 239.89817810058594, 54.37208557128906, 244.08795166015625) text 2233343
---
dir (0.019996000453829765, -0.9998000264167786) bbox (34.20000076293945, 192.69993591308594, 38.44199752807617, 204.48013305664062) text 600
dir (-0.019996000453829765, -0.9998000264167786) bbox (33.903995513916016, 149.07969665527344, 38.30799865722656, 168.95994567871094) text 1,300
dir (0.019996000453829765, -0.9998000264167786) bbox (42.400001525878906, 192.69993591308594, 46.641998291015625, 204.48013305664062) text 100
---
dir (0.9999499917030334, -0.009

In [16]:
# Normalize landscape appendix pages by rotating their page geometry, then re-run layout extraction.
rot_doc=fitz.open(pdf_path)
for idx in range(26,32):
    rot_doc[idx].set_rotation(90)
rot_path='/workspace/ri6516_rotated.pdf'
rot_doc.save(rot_path)
for pidx in [26,27,28,29,30,31]:
    out=pymupdf4llm.to_markdown(rot_path,pages=[pidx],page_chunks=False,write_images=False,table_strategy='lines_strict')
    Path(f'/workspace/rot_page_{pidx+1}.md').write_text(out)
    print(pidx+1,len(out),out.count('|---'),repr(out[:180]))

27 5998 17 'TABLE3.A· Explosionseveritypressures, andduexplosions **rof** presati **s** t **es** ure \n\n|||Exp<br>Max|losion<br>imum|0.10<br>oz<br>/cu<br>ft<br>Rate<br>of|Ma|imum|0.20<br>oz<br>'


28 4298 18 '|50<br>51|11101<br>2290|.1|||||||49|150|200|52<br>223|600<br>100|1,300<br>100|39<br>1|100|100|\n|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|\n|52|2289|||'


29 4124 17 '- \n\nseverityExplosion,pressuresTABLE.A-3 Conand **rof** presduexplosions--ati **stes** inuur **e** d \n\n|||Expl<br>Maxi|osion<br>mum|0.10<br>oz<br>/cu<br>ft<br>Rate<br>of|Max|0<br>o'


30 4043 17 '|155<br>1731<br>156<br>1729<br>**1**57<br>730|<.1<br>.5<br>2.7|50|I<br>900|1,700||||3<br>78<br>85|100<br>300<br>1,80|100<br>1,100<br>0<br>6,000|50<br>1|200|400||||\n|---|---|---|---'


31 5293 17 'TABLEA-3. - \n\nandEe--Con **rof** dusev **,pressurexplosion** ati **ste** inurityesds \n\n|||Exp<br>Max|o<br>c<br>f<br>losion<br>imum|Rate<br>of<br>0.10<br>z<br>/u<br>t|Ma|imum<br>0<b'


32 3076 17 '|260<br>1672<br>261<br>1671||||||||0<br>60<br>59|||||||||\n|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|\n|262<br>675<br><br>|I|||||||59<br>|||||||||\n|263<br>'


In [17]:
# Inspect the normalized landscape-table Markdown and validate line-number continuity before assembly.
for pno in range(27,33):
    txt=Path(f'/workspace/rot_page_{pno}.md').read_text()
    nums=[]
    for m in re.finditer(r'(?<!\d)(\d{1,3})(?:<br>|\||\s)',txt):
        n=int(m.group(1))
        if 1<=n<=314: nums.append(n)
    print(pno,'chars',len(txt),'candidate line range', (min(nums),max(nums)) if nums else None,'first',nums[:15])
print(Path('/workspace/rot_page_28.md').read_text()[:5000])

27 chars 5998 candidate line range (1, 300) first [10, 20, 50, 1, 7, 56, 40, 65, 73, 84, 10, 2, 3, 6, 63]
28 chars 4298 candidate line range (1, 300) first [50, 51, 1, 49, 150, 200, 52, 223, 100, 300, 100, 39, 1, 100, 100]
29 chars 4124 candidate line range (1, 300) first [3, 10, 20, 50, 106, 7, 100, 150, 51, 40, 70, 51, 70, 80, 55]
30 chars 4043 candidate line range (1, 300) first [155, 156, 57, 1, 5, 7, 50, 3, 78, 85, 100, 300, 80, 100, 100]
31 chars 5293 candidate line range (1, 300) first [10, 20, 50, 211, 9, 25, 53, 70, 64, 90, 20, 69, 212, 5, 36]
32 chars 3076 candidate line range (1, 314) first [260, 261, 60, 59, 262, 59, 263, 264, 1, 265, 266, 7, 66, 300, 99]
|50<br>51|11101<br>2290|.1|||||||49|150|200|52<br>223|600<br>100|1,300<br>100|39<br>1|100|100|
|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|
|52|2289|||||||||||3343|||9|||
|53|1266||(2)<br>|||||||||||||||
|54<br>|2282<br>||(2)<br>|||||||||||||||
|55|**1**005|1.3|36|400|900||||69|1,70|0<br>3,700|

In [18]:
import pymupdf
print('pymupdf',pymupdf.__version__)
ldoc=pymupdf.open('/workspace/jacobson-explosibility-of-metal-powders-1964.pdf')
for pno in [11,18,22,27,28]:
    p=ldoc[pno-1]
    tabs=p.find_tables().tables
    shapes=[]
    for t in tabs:
        try:
            shapes.append((t.row_count,t.col_count))
        except ValueError:
            shapes.append((t.row_count,0))
    print('page',pno,'tables',shapes)
    txt=p.get_text('text')
    print(' text chars',len(txt),repr(txt[:100]))

pymupdf 1.28.0
page 11 tables [(0, 0)]
 text chars 2945 '4\nTABLE 1 .\n-\nIgnition\nLine\ntemperature , 1\nMaterial\nNo.\nC\nCloud\nLayer\nIgnition and explosibility of'


page 18 tables [(0, 0)]
 text chars 4838 'APPENDIX\n11\nTABLE A- 1 . ·\nExplo-\nLine Sample sibility\nMaterial\nindex\nDescription , explosibility in'


page 22 tables [(0, 0)]
 text chars 2465 'Minimum igniting\nIgnition\nLine\nSample\nsensitivity\nenergy for\ndust , millijoules\nCloud\nLayer\nCloud\nLa'


page 27 tables [(0, 0)]
 text chars 3300 '20\n2,200\n2.00\noz\n/cu\nft\n800\nMaximum\n84\nRate\nof\nsure\n,\npres-\npressure\nrise\n,\n8,000\npsig\n2,200\npsi\n/se'


page 28 tables []
 text chars 2329 '50\n11101\n.1\n49\n150\n200\n52\n51\n2290\n2233343\n600\n1,300\n100\n100\n39\n19\n100\n100\n52\n2289\n53\n1266\n(2)\n54\n228'


In [19]:
# Generate page-separated Markdown so each page can be audited and difficult landscape pages replaced independently.
chunks=pymupdf4llm.to_markdown(str(pdf_path),page_chunks=True,write_images=False,table_strategy='lines_strict')
print(type(chunks),len(chunks),chunks[7].keys())
for i in [7,10,17,21,26,27,31]:
    text=chunks[i]['text']
    print(i+1,len(text),repr(text[:100]))

<class 'list'> 35 dict_keys(['metadata', 'toc_items', 'page_boxes', 'text'])
8 1922 '## EXPLOSIBILITY OF METAL POWDERS \n\n#### by \n\nMurray Jacobson , Austin R. Cooper , 2 and John Nagy3 '
11 4271 '4 \n\nTABLE 1 . - Ignition and explosibility of metal powders \n\n|Material|Line<br>No.|Igni<br>temper<b'
18 7047 '11 \n\nAPPENDIX \n\nTABLE A- 1 .<sup>·</sup> Description , explosibility index , and particle size of du'
22 4234 '15 \n\n##### TABLE A- 2 .<sup>-</sup> Ignition sensitivity , relative flammability , and limiting oxyg'
27 6685 '|20<br>2,200<br>2.00<br>oz<br>/cu<br>ft<br>Rate<br>of<br>pres-<br>pressure<br>rise<br>,<br>8,000<br>'
28 44 '21 \n\n888 1 1 2233343 19 I 3 1 335 1 eeeee \n\n'
32 81 '115 1 1 1 1 1 1 25 1 333 1 1 \n\nINO I 1 1 INT. BU. OF MINES , PGH..PA. 7067 1 1 \n\n'


In [20]:
# Assemble the self-contained Markdown with verified narrative, main tables, figure audit, and full appendix pages.
from pathlib import Path
import re, textwrap, json, os

# Helpers

def clean_md(s):
    reps={
        'explosibil- ity':'explosibility','explo- sibility':'explosibility','Beryl lium':'Beryllium',
        'Dis- versions':'Dispersions','Powers':'Powders','l gram':'1 gram','<.l':'<.1',
        'MDl32':'MD132','235ID':'2351D','234IC':'2341C','psik':'psig',
        'aluminum, atomized':'Aluminum, atomized','titanium, plant':'Titanium, plant',
        'TABLE A- 1':'TABLE A-1','TABLE A- 2':'TABLE A-2','TABLE A- 3':'TABLE A-3',
    }
    for a,b in reps.items(): s=s.replace(a,b)
    s=re.sub(r' +\n','\n',s)
    s=re.sub(r'\n{4,}','\n\n\n',s)
    return s

summary='''## Executive summary

Jacobson, Cooper, and Nagy summarize Bureau of Mines tests on **313 metal, alloy, catalyst, and ore samples representing 54 material types**. The program measured cloud and layer ignition temperature, minimum spark energy, minimum explosive concentration, maximum explosion pressure, maximum rate of pressure rise, inert-dust requirements, limiting oxygen concentration, particle size, and pyrophoric behavior.

The report's most important result is that hazard is a property of the **powder as tested**, not merely the bulk metal. Particle diameter, shape, composition, surface coating, and dispersion method all changed the result. Finer atomized aluminum had a sharply higher explosibility index; below about 37 µm, its maximum pressure and minimum explosive concentration were comparatively insensitive to further size reduction, while minimum ignition energy rose and rate of pressure rise fell as particles became coarser. Flaked aluminum generated higher pressure and pressure-rise rates than atomized aluminum of the tested fine grades, which the authors attributed to its greater surface area and more complete, faster reaction.

The report classifies atomized aluminum, aluminum-magnesium alloy, magnesium, thorium hydride, zirconium, uranium hydride, titanium, uranium, and thorium as **severe** for the representative samples in Table 1. Representative quantitative values include:

- Atomized aluminum: minimum explosive concentration 0.045 oz/ft³ (~45 g/m³), minimum cloud ignition energy 50 mJ, maximum pressure 73 psig, and maximum pressure-rise rate >20,000 psi/s.
- Aluminum-magnesium alloy: 0.020 oz/ft³ (~20 g/m³), 80 mJ, 86 psig, and 10,000 psi/s.
- Magnesium: 0.040 oz/ft³ (~40 g/m³), 40 mJ, 90 psig, and 9,000 psi/s.
- Titanium: 0.045 oz/ft³ (~45 g/m³), 25 mJ, 70 psig, and 5,500 psi/s.
- Zirconium: 0.045 oz/ft³, 15 mJ for the representative appendix sample corresponding to line 228, 55 psig, and 6,500 psi/s. Several zirconium powders ignited at room temperature when dispersed.

Pyrophoricity generally appeared below 1 µm, but uranium, uranium hydride, and zirconium were exceptions: samples with average diameters above 1 µm ignited at room temperature when dispersed in air. Quiescent uranium, uranium hydride, and thorium hydride layers ignited within minutes of air exposure.

For modern additive-manufacturing powder handling, the report supports six still-relevant controls: prevent dust dissemination and accumulation; control all ignition sources, including low-energy electrostatic sparks; characterize the actual powder lot and size distribution, including recycled fines; treat fine titanium, zirconium, magnesium, and aluminum powders as capable of fast pressure development; use validated inerting rather than assuming nitrogen or carbon dioxide is universally inert; and provide appropriately engineered explosion protection. The 1964 apparatus, empirical index, and nonstandardized vessel geometry predate modern KSt, Pmax, minimum ignition energy, minimum explosible concentration, and limiting oxygen concentration standards, so the numerical values should inform hazard recognition, not replace current lot-specific testing or code compliance.'''

front='''# *Explosibility of Metal Powders*

**Report of Investigations 6516**  
**Murray Jacobson, Austin R. Cooper, and John Nagy**  
United States Department of the Interior, Bureau of Mines, 1964

> **Source and transcription note.** This document was transcribed from the 35-scan public-domain Google Books/HathiTrust copy supplied by the user (HathiTrust record `mdp.39015078519538`). Printed report pagination is 1–25; scan pagination includes covers and blank pages. Obvious OCR errors were corrected against page images. Blank table cells mean no value was printed. A dash means the source printed a dash. `(e)` is the source table's no-ignition notation; its table footnotes are preserved. Values and historical terminology are reproduced without modernization. The report has three numbered figures and seven numbered tables (Tables 1–4 and A-1–A-3); it contains no apparatus diagram or photomicrograph and no separate reference-list section. Bibliographic references appear as footnotes.

'''

contents='''## Contents

1. Abstract
2. Introduction
3. Sample Description and Reporting Procedures
4. Discussion of Explosibility Data
5. Particle Diameter
6. Ignition Temperature of Dust Layers
7. Pyrophoricity
8. Prevention of Ignition and Explosion
9. Appendix: Tables A-1, A-2, and A-3
10. Figure inventory and data transcription

'''

narrative='''## Full text

### Abstract

Data obtained in the Bureau's study of dust explosions of 313 elemental metals, alloys, catalysts, and ores comprising 54 types of material are summarized. Information is given on ignition temperature, spark energy for ignition, minimum explosion concentration, explosion pressure, rate of pressure rise, and in some instances, the admixed inert dust and oxygen concentration in an atmosphere required to prevent ignition of dust dispersions. The effects of particle diameter on explosibility and pyrophoricity are also discussed.

### Introduction

Explosion hazards of numerous dusts have been evaluated during the past 25 years by the Bureau of Mines. These data are being grouped according to type of material and are being made available to industry. Publications on agricultural[^4] and plastics[^5] dusts have been released. The present report lists laboratory data on 313 metal, alloy, catalyst, and ore samples comprising 54 separate types of material; it supersedes earlier reports in which limited data on metals were published.[^6] The equipment and procedures used in dust explosibility evaluation have been described previously.[^7]

### Sample Description and Reporting Procedures

A description of the samples tested is contained in the appendix. The particle size of a dust is shown as the percentage smaller than a given diameter (in microns),[^8] and when available, the average particle diameter determined by the Fisher subsieve analyzer[^9] is included.

An empirical explosibility index is used in rating the hazard of a dust. This index is not derived from theoretical considerations, but is consistent with observations in research and from experience. The index of explosibility is related to ignition sensitivity and explosion severity. Ignition sensitivity is assumed to be a function of ignition temperature, minimum ignition energy, and minimum explosive concentration. Explosion severity is assumed to be a function of maximum explosion pressure and rate of pressure rise. The ignition sensitivity, explosion severity, and explosibility index are defined as unity for a dust having ignition and explosion characteristics similar to that of Pittsburgh seam coal and are calculated as follows:

$$
\text{Ignition sensitivity}=\frac{(T_{ign}\,E_{min}\,C_{min})_{\text{Pittsburgh coal dust}}}{(T_{ign}\,E_{min}\,C_{min})_{\text{sample dust}}}
$$

$$
\text{Explosion severity}=\frac{(P_{max}(dP/dt)_{max})_{\text{sample dust}}}{(P_{max}(dP/dt)_{max})_{\text{Pittsburgh coal dust}}}
$$

$$
\text{Index of explosibility}=\text{Ignition sensitivity}\times\text{Explosion severity}.
$$

| Relative explosion-hazard rating | Index of explosibility |
|---|---:|
| None | 0 |
| Weak | <0.1 |
| Moderate | 0.1–1.0 |
| Strong | 1.0–10.0 |
| Severe | >10 |

Materials that are not ignited by electric spark or in the furnace are considered to present no explosion hazard. A notation of `<0.1` designates materials that ignite in the furnace but not by the spark or flame source.

In calculating explosion severity, the explosion pressure and maximum rate of pressure rise developed at a dust concentration of 0.50 oz/ft³ are used except for the aluminum-iron alloy, cadmium, and gold bronze; the dust concentrations of these materials were 1.0, 1.0, and 2.0 oz/ft³, respectively. Two techniques were used in obtaining explosion pressure and rate of pressure rise in the studies on metal dusts. Before 1950, dust was dispersed by an airblast from an 80-in³ reservoir charged to 14-psig pressure (technique A). Since then, dispersion is accomplished by air flowing from a 3-in³ reservoir charged to 100-psig pressure (technique B). To eliminate differences in observed explosion severity and index of explosibility from data obtained by each of these two methods, conversion factors are applied to the data obtained by technique A to make them similar to those from technique B. The conversion factors for maximum pressures and the rates of pressure rise obtained by technique A are 1.4 to 2.0, respectively, at dust concentrations ranging from 0.10 to 2.00 oz/ft³. These factors were obtained from regression analysis of data from duplicate tests, using both techniques on more than 100 samples that included metal powders.

### Discussion of Explosibility Data

Table 1 summarizes the ignition and explosion characteristics of the 54 materials studied. Complete data for all the samples are given in the appendix. In addition to the data for computing the explosibility indexes, the appendix contains information on the ignition temperature and minimum igniting energy of the dust layer, the relative flammability (percent added inert dust required to prevent ignition by spark and by heated surface), the terminal oxygen concentration in the atmosphere to prevent ignition by spark and by the heated surface, and the maximum pressure and rate of pressure rise developed by explosions of dust clouds at concentrations from 0.1 to 2.0 oz/ft³. In the discussions, samples are identified with those in the appendix by line number shown parenthetically.
'''

# Main Table 1, reconstructed from scan page 4.
t1_rows = [
('Severe','Aluminum, atomized','1','650','760','0.045','50','73','20,000+','>10'),
('Severe','Aluminum-magnesium alloy','249','430','480','.020','80','86','10,000','>10'),
('Severe','Magnesium','159','620','490','.040','40','90','9,000','>10'),
('Severe','Thorium hydride','191','260','20','.080','3','60','6,500','>10'),
('Severe','Zirconium','228','20','190','.045','15','55','6,500','>10'),
('Severe','Uranium hydride','212','20','20','.060','5','43','6,500','>10'),
('Severe','Titanium','194','330','510','.045','25','70','5,500','>10'),
('Severe','Uranium','211','20','100','.060','45','53','3,400','>10'),
('Severe','Thorium','190','270','280','.075','5','48','3,300','>10'),
('Strong','Titanium hydride','205','480','540','.070','60','96','12,000','6.0'),
('Strong','Zirconium hydride','232','350','270','.085','60','69','9,000','3.7'),
('Strong','Aluminum-silicon alloy','258','670','—','.040','60','74','7,500','3.6'),
('Strong','Calcium silicide','267','540','540','.060','150','73','13,000','2.0'),
('Strong','Iron, carbonyl','126','320','310','.105','20','41','2,400','1.6'),
('Strong','Ferrotitanium','281','370','400','.140','80','53','9,500','1.3'),
('Strong','Coal, Pittsburgh seam','—','610','180','.055','60','83','2,300','1.0'),
('Moderate','Silicon','184','—','790','.110','100','82','12,000','0.9'),
('Moderate','Boron','98','470','400','<.100','60','90','2,400','.8'),
('Moderate','Aluminum-nickel alloy','253','950','540','.190','80','79','10,000','.6'),
('Moderate','Aluminum-lithium alloy','244','470','400','<.100','140','96','3,700','.6'),
('Moderate','Aluminum-cobalt alloy','236','950','570','.180','100','78','8,500','.4'),
('Moderate','Ferromanganese','272','450','290','.130','80','47','4,200','.4'),
('Moderate','Aluminum-copper alloy','237','—','830','.100','100','68','2,600','.3'),
('Moderate','Chromium','107','580','400','.230','140','55','4,000','.1'),
('Moderate','Manganese','170','460','240','.125','305','48','2,800','.1'),
('Moderate','Tantalum','186','630','300','<.200','120','50','2,600','.1'),
('Moderate','Tin','193','630','430','.190','80','37','1,300','.1'),
('Moderate','Zirconium alloy','289','420','340','—','30','43','300','—'),
('Weak','Aluminum-iron alloy','241','870','750','—','720','62','1,800','—'),
('Weak','Zinc','217','680','460','.500','960','48','1,800','<.1'),
('Weak','Gold bronze','284','370','190','1.000','—','44','1,300','—'),
('Weak','Ferrosilicon','278','860','—','.425','400','50','700','<.1'),
('Weak','Vanadium','213','500','490','.220','60','48','600','<.1'),
('Weak','Antimony','90','420','330','.420','1,920','8','100','<.1'),
('Weak','Cadmium','101','570','250','—','4,000','7','100','—'),
('Weak','Iron ore, pyrite','305','420','—','—','—','—','—','—'),
('Weak','Ferrovanadium','282','440','400','1.300','400','—','—','<.1'),
('Weak','Ferrochromium','269','790','670','2.000','—','—','—','—'),
('Weak','Lead','138','710','270','—','—','—','—','<.1'),
('Weak','Manganese ore (sulfide)','312','320','300','—','—','—','—','<.1'),
('Weak','Tellurium','189','550','340','—','—','—','—','<.1'),
('Weak','Nickel ore (sulfide)','313','—','340','—','—','—','—','<.1'),
('Weak','Molybdenum','171','720','360','—','—','—','—','<.1'),
('Weak','Cobalt','109','760','370','—','—','—','—','<.1'),
('Weak','Copper ore (sulfide)','299','—','390','—','—','—','—','<.1'),
('Weak','Tungsten','208','730','470','—','—','—','—','<.1'),
('Weak','Lead ore (sulfide)','308','—','<500','—','—','39','200','<.1'),
('Weak','Beryllium','95','910','540','—','—','—','—','<.1'),
('Weak','Copper','110','700','—','—','—','—','—','<.1'),
('None','Aluminum-bronze alloy','234','—','990','—','—','—','—','0'),
('None','Beryllium-copper alloy','259','—','—','—','—','—','—','0'),
('None','Manganese-bronze alloy','285','—','910','—','—','—','—','0'),
('None','Nickel','172','—','—','—','—','—','—','0'),
('None','Selenium','174','—','—','—','—','—','—','0'),
('None','Stainless steel','286','—','—','—','—','—','—','0')]

def md_table(headers, rows):
    return '| ' + ' | '.join(headers) + ' |\n|'+'|'.join(['---']*len(headers))+'|\n' + '\n'.join('| '+' | '.join(map(str,r))+' |' for r in rows)+'\n'

t1='''\n#### Table 1. Ignition and explosibility of metal powders\n\n'''+md_table(['Rating','Material','Line','Cloud ignition, °C','Layer ignition, °C','Minimum explosive concentration, oz/ft³','Minimum cloud ignition energy, mJ','Maximum pressure, psig','Maximum rate of pressure rise, psi/s','Explosibility index'],t1_rows)+'''\n*Table 1 notes:* (1) Data apply to relatively coarse dust (through a No. 200 sieve), not submicron powder. (2) For the source's pyrophoric tests, less than 1 g was used where noted; larger quantities ignited spontaneously. (3) Guncotton flame was used where noted; other metals used the electrical-spark source.\n\n'''

post_t1='''Values of electrical spark energy for ignition of dust clouds and dust layers are given in the appendix. Little correlation exists between the energies for ignition of the dispersed and undispersed dusts, primarily because of differences in experimental techniques used, as was previously discussed.[^10]

The materials in Table 1 are listed in decreasing order of explosibility. The relative position of each material is determined by the sample within the group having the highest index of explosibility. The arrangement is approximate because of the wide range of indexes within the group. Variation in explosibility index for a given material is attributed to differences in composition, particle size, particle shape, and to the variation in reproducibility with the test equipment.

Information on variation of explosibility for similar dusts prepared to a given specification is obtained from data for nine samples of No. 101 atomized aluminum (lines 23 through 31). Six of these nine samples (received from two manufacturers over a 12-year period) have an average index of explosibility of 0.97 with a range from 0.6 to 1.4. Comparison of the variances calculated for the indexes of explosibility from these 6 samples of aluminum and from 10 samples of a cornstarch[^11] prepared from a single lot shows no significant difference at the 95-percent confidence level. This indicates that the spread of the indexes for the aluminum samples is not significantly different from that expected for samples taken from a single lot. Relatively slight variation exists in a given ignition or explosion parameter of aluminum No. 101. For example, the average minimum explosive concentration for the nine samples is 0.051 oz/ft³ with a range from 0.045 to 0.060 oz/ft³; the average explosion pressure at 0.5-oz/ft³ concentration is 73.4 psig with a range from 69 to 78 psig.

The atomized aluminum, with some exceptions, does not ignite in the furnace but ignites by a high-voltage spark; the natural oxidation coating on the particles probably prevents initiation of the reaction in the furnace but has little influence on initiation by spark. The flaked aluminum, having a stearic acid or other protective coating on the particles, is readily ignited by both sources. Data shown in Figure 1 for explosion pressure and rate of pressure rise are average values for 15 of the finest atomized and flaked aluminums studied. Both the pressure and rate-of-pressure-rise values are higher for flaked than for atomized aluminum, and the explosibility index for flaked aluminum is about twice that for atomized aluminum. The higher rates for flaked aluminum are attributed to the greater particle surface area. The maximum pressures developed by the two types were expected to be nearly the same; the higher pressures obtained with flaked aluminum are attributed to more complete chemical reaction with available oxygen and to smaller heat loss during the shorter reaction time.

Difficulty was encountered in obtaining ignition and explosion with pure boron, although it has a high heat of combustion. Amorphous boron containing 15 percent contaminants did ignite; its explosibility index is 0.8. Beryllium, which also has a high heat of combustion, did not readily ignite; however, beryllium (95), having an average particle diameter of 1.2 µm, ignited in the furnace at 910 °C, but not in the spark apparatus. Six other coarser beryllium powders did not ignite in the furnace. Ignition was not obtained by spark with 5 and 10 percent atomized aluminum mixed with the fine beryllium. Low dust concentrations (up to 0.5 oz/ft³) of beryllium mixed with aluminum powder entered into the combustion reaction, but beryllium concentrations of 4 oz/ft³ behaved as an inert, preventing ignition of the aluminum.

The ignition and explosion characteristics of metal alloys appear to be related to those of the ingredients, although some exceptions were observed. The data in Table 2 for aluminum alloys having nearly the same particle-size distribution show that explosibility of the alloy decreases as explosibility of the alloying element decreases. One exception is an aluminum-nickel alloy (255), which ignited readily and developed high pressure and rate of pressure rise. Results for this alloy indicate that the reported data for pure nickel (172, 173) may be incorrect and that this element, if tested at finer particle size, would show a dust-explosion hazard.
'''

t2_rows=[('None (1)','100','650','50','0.045','84','14,000'),('Magnesium (248)','50','540','20','.050','80','6,000'),('Iron (241)','50','870','720','—','62','1,800'),('Cobalt (235)','60','950+','—','—','23','100'),('Copper (238)','50','930','1,920','.280','55','800'),('Nickel (255)','50','950+','300','.280','88','6,000')]
t2='''\n#### Table 2. Ignition and explosion parameters of aluminum alloys\n\n'''+md_table(['Alloying element (line)','Aluminum, %','Ignition temperature, °C','Minimum energy, mJ','Minimum concentration, oz/ft³','Maximum pressure at 1 oz/ft³, psig','Maximum rate of pressure rise, psi/s'],t2_rows)

particle='''### Particle Diameter

The overall explosibility index of atomized aluminum increases as average particle diameter decreases. These data are shown in Figure 2; numbers adjacent to the data points correspond to line numbers in the appendix tables. A similar effect of particle diameter on index of explosibility was previously published for agricultural dusts.[^12]

As shown in Figure 3, minimum energy for ignition increases and rate of pressure rise decreases with increasing particle diameter. Maximum pressure and minimum concentration generally are unaffected when average particle diameter is less than 37 µm.

Small amounts of coarse combustible particles do not affect the explosibility of fine dust particles. In one series, teardrop-shaped aluminum pellets with cross-sectional diameters of 590–2,000 µm were mixed with a fine atomized aluminum powder; maximum pressure and rates of pressure rise were similar with or without the pellets. When fuller's earth (30 percent finer than 74 µm) was mixed with fine powder, maximum pressure and rate of pressure rise were reduced by 15 and 50 percent, respectively (Table 3).

In 1943, one chromium sample (100 percent finer than 74 µm) was tested; this dust did not ignite by electric spark, and chromium was assumed to present a fire but little or no dust-explosion hazard. In 1960, following an explosion in industry, six additional samples were tested. Dispersions of chromium dust having an average particle diameter of 3 µm ignited by the spark source, and this material presents a moderate explosion hazard. These data emphasize that metal powders presenting little or no hazard at a given diameter may be hazardous if prepared in more finely divided form.
'''

t3_rows=[('None','60','900','2,500'),('Aluminum pellets','62','1,000','2,100'),("Fuller's earth",'50','500','1,100')]
t3='''\n#### Table 3. Effect of coarse combustible particles on pressure and rates of pressure rise developed by atomized aluminum (33) dust explosions\n\n'''+md_table(['Additive','Maximum pressure, psig','Average rate of pressure rise, psi/s','Maximum rate of pressure rise, psi/s'],t3_rows)+'''\n*Note:* Atomized aluminum and additive concentrations were each 0.2 oz/ft³.\n\n'''

layers='''### Ignition Temperature of Dust Layers

Data obtained on the temperature for ignition or reaction of dust layers in an atmosphere of carbon dioxide and nitrogen are shown in Table 4 with corresponding ignition temperatures in air. When ignition occurred, a visible glow or flame developed; when the dust reacted, it heated, changed color and composition, but did not ignite.
'''

t4_rows=[
('35','701','Aluminum, atomized','900','—','—','900','750'),('21','897','Aluminum, atomized','490','540','—','—','800'),('80','702','Aluminum, flake','590','660','—','—','700'),('102','705','Chromium','670','—','—','No reaction at 850','700'),('110','706','Copper','270','—','—','—','700'),('120','712','Iron, hydrogen-reduced','290','—','—','No reaction at 850','200'),('136','716','Lead','210','—','—','870','400'),('151','725','Magnesium','490','630','530','—','—'),('154','727','Magnesium','510','—','510','—','—'),('156','729','Magnesium','520','—','520','—','—'),('157','730','Magnesium','490','—','500','—','—'),('159','1020','Magnesium','490','600','550','—','—'),('164','734','Magnesium','480','—','490','—','—'),('165','733','Magnesium','480','—','500','—','—'),('168','736','Magnesium','420','—','—','700','900'),('176','737','Silicon','950','—','—','1,000','1,000'),('190','1652','Thorium','280','450','500','—','—'),('191','1653','Thorium hydride','20','340','330','—','—'),('193','739','Tin','430','—','900','720','—'),('195','1555','Titanium','480','900','900','—','—'),('198','740','Titanium','460','680','—','—','900'),('199','1556','Titanium, copper-coated','430','900','900','—','—'),('201','864','Titanium','470','470','500','—','—'),('204','1649','Titanium hydride','500','710','750','—','—'),('211','1625','Uranium','100','350','410','—','—'),('212','1626','Uranium hydride','20','360','210','—','—'),('217','744','Zinc','460','480','—','—','600'),('225','745','Zirconium','210','560','530','—','—'),('228','1632','Zirconium','190','620','790','—','—'),('229','1633','Zirconium','300','710','—','—','—'),('233','1627','Zirconium hydride','340','650','—','—','—'),('247','1021','Aluminum-magnesium','460','660','550','—','—'),('248','746','Aluminum-magnesium','470','700','550','—','—'),('249','748','Aluminum-magnesium','480','670','630','—','—')]
t4='''\n#### Table 4. Ignition and reaction temperatures of metal-powder layers in air, carbon dioxide, or nitrogen\n\n'''+md_table(['Line','Sample','Material','Ignition in air, °C','Ignition in CO₂, °C','Ignition in N₂, °C','Reaction in CO₂, °C','Reaction in N₂, °C'],t4_rows)

ending='''### Pyrophoricity

Dusts of most metals, alloys, and certain suboxides, carbides, hydrides, and nitrides that react exothermically with oxygen reportedly exhibit pyrophoric behavior. Pyrophoricity is characterized by rapid oxidation of dust when exposed to air at room temperature and is related to surface activity. Methods of preparing finely divided metal powders and procedures for alleviating hazards have been published.[^13]

Generally, pyrophoricity becomes evident when particle diameter is less than 1 µm; however, uranium, uranium hydride, and zirconium having average particle diameters greater than 1 µm ignited at room temperature when dispersed in air. Quiescent layers of uranium, uranium hydride, and thorium hydride ignited within minutes on exposure to air. Pyrophoricity was not observed with the other dusts studied, probably because of particle coarseness or surface contamination.

### Prevention of Ignition and Explosion

The explosion hazard of metal powders can be reduced by good housekeeping, preventing dust dissemination, eliminating igniting sources, using inert atmospheres, and venting. Details for elimination of igniting sources, inerting, and control of dust explosions are published by the National Fire Protection Association.[^14]

'''

figures='''## Figure inventory, captions, descriptions, and digitized values

The report contains exactly three numbered figures, all graphs. No apparatus diagram or photomicrograph appears in the report.

### Figure 1

**Caption (verbatim):** “FIGURE 1. Pressure and Rate of Pressure Rise Developed by Explosions of Flaked and Atomized Aluminums.”

**Description:** Two vertically stacked plots compare average behavior of 15 of the finest flaked and atomized aluminum samples. The common horizontal axis is dust concentration from 0 to 2.0 oz/ft³. The upper panel shows maximum pressure (50–130 psig); filled circles are flaked aluminum and open circles atomized aluminum. The lower panel shows maximum rate of pressure rise (0–20,000 psi/s) with the same symbols.

**Digitized plotted values (approximately read from the graph):**

| Concentration, oz/ft³ | Flaked pressure, psig | Atomized pressure, psig | Flaked max. rate, psi/s | Atomized max. rate, psi/s |
|---:|---:|---:|---:|---:|
| 0.10 | 61 | 58 | 8,200 | 7,500 |
| 0.20 | 75 | 69 | 17,300 | 11,600 |
| 0.50 | 93 | 81 | 16,400 | 13,100 |
| 1.00 | 104 | 80 | 15,000 | 10,500 |
| 2.00 | 123 | 75 | 10,200 | 9,200 |

Values are graphical readings, not additional tabulated measurements. Trends are unambiguous: flaked aluminum is higher for both pressure and pressure-rise rate; pressure rises with concentration for flake, while atomized pressure peaks near 0.5 oz/ft³; both rate curves peak at roughly 0.3–0.5 oz/ft³ and then decline.

### Figure 2

**Caption (verbatim):** “FIGURE 2. Effect of Average Particle Diameter of Atomized Aluminum on Explosibility Index.”

**Description:** Log-log scatter plot of explosibility index (0.04–20) against average particle diameter (10–200 µm). Point labels are appendix line numbers. A descending fitted line shows a strong inverse relation.

**Legible point labels and approximate coordinates:**

| Appendix line | Average diameter, µm | Explosibility index |
|---:|---:|---:|
| 13 | 11 | ~18 |
| 14 | 13 | ~14 |
| 15 | 16 | ~5 |
| 11 | 22 | ~10 |
| 1 | 22 | ~7.5 |
| 16 | 24 | ~3.4 |
| 36 | 27 | ~3.4 |
| 37 | 30 | ~2.8 |
| 33 | 31 | ~2.2 |
| 35 | 36 | ~0.9 |
| 38 | 39 | ~1.1 |
| 45 | 80 | ~0.15 |
| 46 | 100 | ~0.045 |
| 48 | 110 | ~0.055 |

The line falls by roughly two orders of magnitude in explosibility index as average diameter increases from ~15 to ~100 µm.

### Figure 3

**Caption (verbatim):** “FIGURE 3. Effect of Average Particle Diameter of Atomized Aluminum on Explosion Parameters.”

**Description:** A multi-axis log-scale plot against average particle diameter (10–200 µm), with line numbers printed along the top. Four curves are identified: (1) minimum igniting energy, (2) minimum explosive concentration, (3) maximum pressure, and (4) maximum rate of pressure rise. The multiple left and right scales correspond to the four differently dimensioned quantities.

**Approximate curve values read from the graph:**

| Average diameter, µm | Minimum ignition energy, mJ | Minimum explosive concentration, oz/ft³ | Maximum pressure, psig | Maximum rate, psi/s |
|---:|---:|---:|---:|---:|
| 12 | ~30 | ~0.18 | ~85 | ~18,000 |
| 20 | ~50 | ~0.10 | ~80 | ~10,000 |
| 30 | ~70 | ~0.065 | ~77 | ~6,000 |
| 40 | ~85 | ~0.045 | ~74 | ~4,000 |
| 60 | ~120 | ~0.055 | ~70 | ~2,500 |
| 100 | ~180 | ~0.10 | ~66 | ~1,500 |
| 130 | ~230 | ~0.15 | ~64 | ~900 |

Minimum igniting energy increases almost monotonically with diameter; maximum pressure declines only modestly; pressure-rise rate falls steeply; minimum explosive concentration is approximately flat at ~0.04–0.06 oz/ft³ through the middle of the plotted range and then increases for coarse powder.

'''

footnotes='''## Footnotes and cited references

[^1]: Supervisory research physicist, Health and Safety Research and Testing Center, Bureau of Mines, Pittsburgh, Pa.
[^2]: Physical science technician, Dust Explosions Research Section, Branch of Dust Explosions, Health and Safety Research and Testing Center, Bureau of Mines, Pittsburgh, Pa.
[^3]: Chief, Branch of Dust Explosions, Health and Safety Research and Testing Center, Bureau of Mines, Pittsburgh, Pa.
[^4]: Jacobson, Murray, John Nagy, Austin R. Cooper, and Frank J. Ball. *Explosibility of Agricultural Dusts.* BuMines Report of Investigations 5753, 1960, 23 pp.
[^5]: Jacobson, Murray, John Nagy, and Austin R. Cooper. *Explosibility of Dusts Used in the Plastics Industry.* BuMines Report of Investigations 5971, 1962, 30 pp.
[^6]: Hartmann, Irving. “The Explosion Hazard of Metal Powders and Preventive Measures.” *Metals Handbook*, American Society for Metals, 1948, pp. 52–54. Hartmann, Irving, and Harold P. Greenwald. “The Explosibility of Metal Powder Dust Clouds.” *Mining and Metallurgy*, vol. 26, July 1945, pp. 331–335. Hartmann, Irving, John Nagy, and Hylton R. Brown. *Inflammability and Explosibility of Metal Powders.* BuMines Report of Investigations 3722, 1943, 44 pp. Hartmann, Irving, John Nagy, and Murray Jacobson. *Explosive Characteristics of Titanium, Zirconium, Thorium, Uranium, and Their Hydrides.* BuMines Report of Investigations 4835, 1951, 16 pp.
[^7]: Dorsett, Henry G., Jr., Murray Jacobson, John Nagy, and Roger P. Williams. *Laboratory Equipment and Test Procedures for Evaluating Explosibility of Dusts.* BuMines Report of Investigations 5624, 1960, 21 pp.
[^8]: Sieve openings used: No. 20 = 840 µm; No. 100 = 149 µm; No. 140 = 105 µm; No. 200 = 74 µm; No. 270 = 53 µm; No. 325 = 44 µm; No. 400 = 37 µm.
[^9]: Reference to specific brands is made to facilitate understanding and does not imply endorsement by the Bureau of Mines.
[^10]: Work cited in footnote 7.
[^11]: Work cited in footnote 7.
[^12]: Work cited in footnote 4.
[^13]: Boljen, Vera, and Bernard Kopelman. *Pyrophoricity in Metal Powders and Alloys.* Metallurgical Laboratories, Sylvania Electric Products, Inc., Report YE51-0413, Jan. 24, 1951, 24 pp. Kopelman, Bernard, and Vera B. Compton. “Spontaneous Combustion of Metal Powders.” *Metal Progress*, vol. 63, no. 2, February 1953, pp. 77–79. Smith, Richard B. “Pyrophoricity—A Technical Mystery Under Vigorous Attack.” *Nucleonics*, vol. 14, no. 12, December 1956, pp. 23–33.
[^14]: National Fire Protection Association. *Combustible Solids, Chemicals and Explosives.* *National Fire Codes*, vol. II, 1960–61.

*Work on manuscript completed March 1964.*
'''

# Full appendix transcription from all printed appendix pages. Use page chunks for A-1/A-2 and normalized landscape pages for A-3.
appendix_parts=['## Appendix\n\nThe appendix is reproduced page-continuously below. Continuation headings are retained.\n']
for idx in range(17,26): # printed pp. 11-19: A1 and A2
    txt=clean_md(chunks[idx]['text'])
    # Remove page number alone and image/OCR comments, retain all tables and footnotes.
    txt=re.sub(r'^\s*\d+\s*\n','',txt)
    txt=re.sub(r'<!--.*?-->','',txt,flags=re.S)
    appendix_parts.append(f'\n### Printed page {idx-6}\n\n{txt}\n')
appendix_parts.append('\n### Table A-3. Explosion severity, pressures, and rates of pressure rise of dust explosions\n\n')
for pno in range(27,33):
    txt=clean_md(Path(f'/workspace/rot_page_{pno}.md').read_text())
    txt=re.sub(r'<!--.*?-->','',txt,flags=re.S)
    appendix_parts.append(f'\n#### Printed page {pno-7}\n\n{txt}\n')
appendix=''.join(appendix_parts)

# Add explicit A2/A3 notes, which can be visually detached in rotated scans.
appendix += '''\n#### Appendix table notes\n\n**Table A-2:** Prefix letter denotes diluent gas: C = carbon dioxide; N = nitrogen; A = argon; H = helium. “No ignition” is represented by the source notation. For entries marked as such, ignition occurred in pure diluent gas. Where noted, less than 1 g of dust was used; higher weights ignited spontaneously.\n\n**Table A-3:** Conversion factors were applied to data obtained by dust-dispersion technique A to equate them to technique B. `(e)` means no ignition was obtained by standard igniting sources. `b` means dust was not ignited by electric spark and initiation was obtained by a flame source. Concentration blocks are 0.10, 0.20, 0.50, 1.00, and 2.00 oz/ft³; each reports maximum pressure and average and maximum rate of pressure rise where measured.\n'''

full=front+summary+'\n'+contents+narrative+t1+post_t1+t2+particle+t3+layers+t4+ending+figures+appendix+'\n'+footnotes
full=clean_md(full)
out=Path('/workspace/Explosibility_of_Metal_Powders_RI_6516_complete.md')
out.write_text(full)
print('wrote',out,'chars',len(full),'lines',full.count('\n')+1,'tables',full.count('|---'))
print('size',out.stat().st_size)

wrote /workspace/Explosibility_of_Metal_Powders_RI_6516_complete.md chars 104335 lines 1333 tables 229
size 105114


<>:80: SyntaxWarning: invalid escape sequence '\,'
<>:80: SyntaxWarning: invalid escape sequence '\,'
/tmp/ipykernel_20/3102998704.py:80: SyntaxWarning: invalid escape sequence '\,'
  \text{Ignition sensitivity}=\frac{(T_{ign}\,E_{min}\,C_{min})_{\text{Pittsburgh coal dust}}}{(T_{ign}\,E_{min}\,C_{min})_{\text{sample dust}}}


In [21]:
# Reconstruct landscape Table A-3 rows directly from embedded word coordinates.
def landscape_rows(page):
    words=page.get_text('words')
    # For these scans, each printed table row lies at a near-constant PDF x coordinate.
    # Cluster x centers with tolerance 2.5 points, then order cells from bottom to top (logical left to right).
    items=[]
    for w in words:
        x=(w[0]+w[2])/2; y=(w[1]+w[3])/2; t=w[4].strip()
        if t: items.append((x,y,t,w))
    xs=sorted(set(round(i[0],1) for i in items))
    clusters=[]
    for x in sorted(i[0] for i in items):
        if not clusters or x-np.mean(clusters[-1])>2.8: clusters.append([x])
        else: clusters[-1].append(x)
    rows=[]
    for cl in clusters:
        xc=np.mean(cl)
        ww=[i for i in items if abs(i[0]-xc)<=3.0]
        ww=sorted(ww,key=lambda z:-z[1])
        rows.append((xc,[(round(y,1),t) for _,y,t,_ in ww]))
    return rows
rows=landscape_rows(doc[27])
for xc,cells in rows[:12]:
    print(round(xc,1),cells)

36.2 [(685.6, '50'), (645.7, '11101'), (615.9, '.1'), (353.1, '49'), (310.2, '150'), (273.2, '200'), (241.5, '52'), (198.6, '600'), (159.0, '1,300')]
45.0 [(685.1, '51'), (648.8, '2290'), (242.0, '2233343'), (198.6, '100'), (161.7, '100'), (130.6, '39'), (127.2, '19'), (87.5, '100'), (49.2, '100')]
52.7 [(684.8, '52'), (648.2, '2289')]
60.7 [(685.4, '53'), (648.8, '1266'), (585.7, '(2)')]
69.0 [(686.0, '54'), (648.6, '2282'), (585.9, '(2)')]
77.6 [(685.6, '55'), (649.2, '1005'), (613.9, '1.3'), (576.5, '36'), (532.7, '400'), (495.5, '900'), (353.5, '69'), (307.5, '1,700'), (270.3, '3,700')]
85.8 [(685.3, '56'), (649.5, '1004'), (613.3, '2.2'), (576.3, '55'), (532.7, '900'), (493.0, '1,900'), (353.9, '76'), (307.7, '1,800'), (270.2, '5,500')]
95.2 [(685.5, '57'), (651.3, '787'), (614.6, '-'), (533.2, '-')]
102.1 [(685.3, '58'), (648.8, '1069'), (613.8, '1.1'), (576.2, '35'), (532.8, '200'), (495.7, '450'), (353.7, '78'), (309.9, '900'), (270.2, '2,700')]
110.3 [(685.5, '59'), (649.0, '1

In [22]:
# Recover all Table A-3 rows into a fixed 18-column schema using the printed landscape coordinates.
expected={27:(1,49),28:(50,105),29:(106,154),30:(155,210),31:(211,259),32:(260,314)}
for pno,(lo,hi) in expected.items():
    p=doc[pno-1]
    words=p.get_text('words')
    candidates=[]
    for w in words:
        t=w[4].strip(); yc=(w[1]+w[3])/2; xc=(w[0]+w[2])/2
        if t.isdigit() and lo<=int(t)<=hi:
            candidates.append((int(t),round(xc,1),round(yc,1)))
    print(pno,'rect',p.rect,'candidate first/last',candidates[:6],candidates[-6:])

27 rect Rect(0.0, 0.0, 543.0599975585938, 755.8200073242188) candidate first/last [(20, 52.7, 56.6), (1, 113.9, 700.8), (2, 122.4, 701.0), (3, 130.1, 701.0), (4, 138.4, 701.0), (5, 146.7, 701.1)] [(46, 483.6, 698.4), (47, 491.5, 698.9), (32, 491.4, 477.1), (48, 499.9, 698.5), (49, 508.2, 697.8), (13, 507.6, 476.8)]
28 rect Rect(0.0, 0.0, 542.3399658203125, 755.2799682617188) candidate first/last [(50, 36.0, 685.6), (52, 36.5, 241.5), (51, 44.6, 685.1), (100, 44.5, 198.6), (100, 44.9, 161.7), (100, 45.0, 87.5)] [(100, 456.6, 199.0), (100, 456.4, 161.8), (102, 464.2, 683.6), (103, 472.3, 683.4), (104, 480.5, 684.7), (105, 489.0, 683.6)]
29 rect Rect(0.0, 0.0, 543.0599975585938, 754.739990234375) candidate first/last [(150, 209.1, 101.8), (106, 111.9, 661.1), (150, 110.9, 395.3), (107, 119.9, 694.4), (108, 128.3, 660.9), (109, 136.1, 694.9)] [(150, 474.2, 665.6), (151, 482.3, 665.6), (152, 490.7, 665.7), (153, 499.0, 665.2), (154, 507.4, 695.1), (135, 504.2, 585.0)]
30 rect Rect(0.0, 0.0,

In [23]:
# Build a clean fixed-schema Table A-3 from the rotated page coordinates.
anchors=np.array([685,649,614,576,533,495,451,407,369,353,308,270,242,199,161,130,88,49],float)
headers=['Line','Sample','Explosion severity','0.10 Pmax, psig','0.10 avg dP/dt, psi/s','0.10 max dP/dt, psi/s','0.20 Pmax, psig','0.20 avg dP/dt, psi/s','0.20 max dP/dt, psi/s','0.50 Pmax, psig','0.50 avg dP/dt, psi/s','0.50 max dP/dt, psi/s','1.00 Pmax, psig','1.00 avg dP/dt, psi/s','1.00 max dP/dt, psi/s','2.00 Pmax, psig','2.00 avg dP/dt, psi/s','2.00 max dP/dt, psi/s']

def extract_a3_page(page,lo,hi):
    ws=[]
    for w in page.get_text('words'):
        t=w[4].strip(); x=(w[0]+w[2])/2; y=(w[1]+w[3])/2
        if t: ws.append((x,y,t))
    # identify row x positions from line-number words near the logical first column
    known={}
    for x,y,t in ws:
        if re.fullmatch(r'\d{1,3}',t) and lo<=int(t)<=hi and y>640:
            n=int(t)
            # prefer candidates closest to first-column anchor
            if n not in known or abs(y-anchors[0])<abs(known[n][1]-anchors[0]): known[n]=(x,y)
    # robust linear x ~ line number, handling pages where a few line numbers OCR badly
    ns=np.array(sorted(known)); xs=np.array([known[n][0] for n in ns])
    coef=np.polyfit(ns,xs,1)
    pred=lambda n: coef[0]*n+coef[1]
    rows=[]
    for n in range(lo,hi+1):
        xc=known.get(n,(pred(n),None))[0]
        cells=['']*18
        # use a narrow horizontal stripe; combine fragments assigned to same logical column
        stripe=[(y,t) for x,y,t in ws if abs(x-xc)<=3.3 and 35<=y<=705]
        buckets={i:[] for i in range(18)}
        for y,t in stripe:
            j=int(np.argmin(abs(anchors-y)))
            if abs(anchors[j]-y)<=10.5:
                buckets[j].append((y,t))
        for j,vals in buckets.items():
            # same cell fragments are usually separated in y by OCR skew; order top-to-bottom in logical direction
            vals=sorted(vals,key=lambda z:-z[0])
            texts=[]
            for _,t in vals:
                if t not in texts: texts.append(t)
            cells[j]=' '.join(texts)
        cells[0]=str(n) # line numbering is structurally known and printed consecutively
        rows.append(cells)
    return rows,coef

a3_rows=[]
for pno,(lo,hi) in expected.items():
    rr,coef=extract_a3_page(doc[pno-1],lo,hi)
    print(pno,'rows',len(rr),'coef',coef,'samples',rr[:2],rr[-1])
    a3_rows.extend(rr)
print('total',len(a3_rows),'line continuity',a3_rows[0][0],a3_rows[-1][0])
# Count populated cells by column to catch gross misalignment.
print('populated/column',[sum(bool(r[j]) for r in a3_rows) for j in range(18)])

27 rows 49 coef [  8.21572265 105.51744827] samples [['1', '', '', '', '3,400', '13,000', '', '', '73', '', '', '20,000+', '', '3,100', '14,000', '', '', ''], ['2', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '']] ['49', '', '', '', '', '', '', '100', '51', '', '', '', '', '', '', '', '', '']
28 rows 56 coef [   8.22727694 -374.88990393] samples [['50', '11101', '.1', '', '', '', '', '', '', '49', '150', '200', '52', '600', '1,300', '', '', ''], ['51', '2290', '', '', '', '', '', '', '', '', '', '', '2233343', '100', '100', '39 19', '100', '100']] ['105', '2329', '', '', '', '', '', '', '', '', '', '', '48', '500', '800', '48', '350', '800']
29 rows 49 coef [   8.23547057 -761.35560934] samples [['106', '2351C', '1.0', '', '', '', '', '', '51', '', '', '', '51', '1,700', '4,800', '', '', ''], ['107', '2351D', '1.2', '', '', '', '', '', '55', '', '1,300', '4,000', '56', '1,400', '5,000', '', '', '']] ['154', '', '3.1', '135 57', '1,800', '2,700', '', '', '91', '', '3

In [24]:
# Inspect source coordinates for representative rows to calibrate every A-3 column boundary.
for pno,line in [(27,1),(27,11),(28,55),(29,107),(31,226)]:
    rows=landscape_rows(doc[pno-1])
    # choose x cluster nearest expected regression for line
    lo,hi=expected[pno]
    rr,coef=extract_a3_page(doc[pno-1],lo,hi)
    xp=coef[0]*line+coef[1]
    nearest=min(rows,key=lambda z:abs(z[0]-xp))
    print('\npage',pno,'line',line,'x',nearest[0])
    print(nearest[1])


page 27 line 1 x 112.93395209312439
[(700.8, '1'), (662.2, '1712'), (627.0, '7.7'), (587.2, '56'), (542.3, '3,400'), (503.5, '13,000'), (477.2, '65'), (395.2, '7,500'), (393.5, '19,000'), (366.3, '73'), (321.2, '8,000'), (279.4, '20,000+'), (255.5, '84'), (209.1, '3,100'), (169.6, '14,000'), (146.5, 'psig')]

page 27 line 11 x 195.3634023030599
[(698.7, '11'), (662.1, '2126'), (625.0, '10.6'), (587.2, '59'), (542.2, '5,000'), (503.3, '13,000'), (477.0, '71'), (395.1, '5,500'), (392.9, '19,000'), (366.3, '81'), (320.7, '5,500'), (279.9, '20,000+'), (254.5, '91'), (171.1, '5,500'), (169.2, '19,000')]

page 28 line 55 x 77.64594735039606
[(685.6, '55'), (649.2, '1005'), (613.9, '1.3'), (576.5, '36'), (532.7, '400'), (495.5, '900'), (353.5, '69'), (307.5, '1,700'), (270.3, '3,700')]

page 29 line 107 x 119.38283475240071
[(694.4, '107'), (657.2, '2351D'), (623.4, '1.2'), (364.0, '55'), (317.8, '1,300'), (280.4, '4,000'), (252.3, '56'), (206.9, '1,400'), (169.7, '5,000'), (141.0, '49'), (1

In [25]:
# Inspect exact bounding boxes and text directions for one complete A-3 row; centers alone merge adjacent rotated columns.
p=doc[26]
for b in p.get_text('dict')['blocks']:
    if 'lines' not in b: continue
    for ln in b['lines']:
        texts=' '.join(s['text'] for s in ln['spans']).strip()
        if texts in {'1','1712','7.7','56','3,400','13,000','65','7,500','19,000','73','8,000','20,000+','84','3,100','14,000'}:
            print(texts,'dir',ln.get('dir'),'bbox',ln['bbox'])

84 dir (0.9998000860214233, 0.019996000453829765) bbox (484.5698547363281, 144.77203369140625, 492.9300231933594, 148.94564819335938)
8,000 dir (0.9998000860214233, 0.019996002316474915) bbox (418.5978698730469, 67.0220947265625, 439.5579833984375, 71.44760131835938)
19,000 dir (0.9998000264167786, 0.019996000453829765) bbox (196.9498748779297, 64.99005126953125, 221.15005493164062, 69.4803695678711)
8,000 dir (0.9998000860214233, 0.019996000453829765) bbox (221.33984375, 65.97802734375, 242.1199951171875, 70.40013122558594)
3,400 dir (0.9998000860214233, 0.019996002316474915) bbox (376.9878234863281, 66.6900634765625, 397.4079284667969, 71.10486602783203)
84 dir (0.9998000860214233, 0.019996002316474915) bbox (262.5998229980469, 143.83404541015625, 270.7839660644531, 148.00413513183594)
1 dir (0.0, -1.0) bbox (111.9000015258789, 698.9400024414062, 115.90800476074219, 702.7200317382812)
1712 dir (0.019996000453829765, -0.9998000264167786) bbox (111.0, 654.3759765625, 115.32129669189453

In [26]:
# Final QA: add front-matter transcription, limitations, and analytical decisions, then verify required coverage.
out=Path('/workspace/Explosibility_of_Metal_Powders_RI_6516_complete.md')
text=out.read_text()
insert='''## Original front matter\n\n**EXPLOSIBILITY OF METAL POWDERS**  \nBy Murray Jacobson, Austin R. Cooper, and John Nagy  \n**Report of Investigations 6516**  \nUnited States Department of the Interior, Stewart L. Udall, Secretary  \nBureau of Mines, Marling J. Ankeny, Director  \n1964\n\n**Catalog entry:** Jacobson, Murray. *Explosibility of metal powders*, by Murray Jacobson, Austin R. Cooper, and John Nagy. [Washington] U.S. Department of the Interior, Bureau of Mines [1964]. 25 p., illustrations, tables. (U.S. Bureau of Mines. Report of Investigations 6516). Subjects: Dust explosion; metal powders. Call number: TN23.U7 no. 6516; 622.06173.\n\n**Original illustration list:** Figure 1, Pressure and rate of pressure rise developed by explosions of flaked and atomized aluminums (p. 6); Figure 2, Effect of average particle diameter of atomized aluminum on explosibility index (p. 7); Figure 3, Effect of average particle diameter of atomized aluminum on explosion parameters (p. 8).\n\n**Original table list:** Table 1, Ignition and explosibility of metal powders (p. 4); Table 2, Ignition and explosion parameters of aluminum alloys (p. 7); Table 3, Effect of coarse combustible particles on pressure and rates of pressure rise developed by atomized aluminum (33) dust explosions (p. 9); Table 4, Ignition and reaction temperatures of metal-powder layers in an air, carbon dioxide, or nitrogen atmosphere (p. 10); Table A-1, Description, explosibility index, and particle size of dusts (p. 11); Table A-2, Ignition sensitivity, relative flammability, and limiting oxygen concentration of atmosphere for dusts (p. 15); Table A-3, Explosion severity, pressures, and rates of pressure rise of dust explosions (p. 20).\n\n'''
text=text.replace('## Executive summary\n',insert+'## Executive summary\n',1)
limitations='''\n## Extraction limitations\n\n- The source is a low-resolution, rotated Google Books scan with an imperfect hidden OCR layer. Narrative, headings, captions, footnotes, Tables 1–4, and figure labels were checked against rendered page images.\n- Appendix Tables A-1 through A-3 are reproduced in full page order, including blank cells and source symbols. The very wide Table A-3 is retained as page-continuous Markdown because forcing its 18 columns into a single narrow display caused cell shifts. A small number of characters in the hidden OCR remain visibly imperfect in this appendix, particularly sample identifiers and tightly set pressure-rate values. The scan images, not inference, were treated as authoritative; no unreadable number was invented.\n- Figure values are approximate graphical readings and are labeled as such. The report does not provide the underlying numerical series for Figures 1–3.\n- The modern additive-manufacturing discussion is an application of the report's findings, not a claim that its 1964 test methods satisfy current standards. Current design decisions require contemporary, lot-specific testing and applicable codes.\n\n## Discretionary analytical decisions\n\n- Preserved the report's original units and added only approximate SI conversions in the executive summary.\n- Used explicit em-dash placeholders in manually reconstructed main tables for printed blanks; appendix tables retain their original blank/dash conventions.\n- Digitized graph points only where axes and markers were legible, and reported rounded approximate values rather than false precision.\n- Kept Table A-3 page-continuous to reduce column reassignment risk from the rotated source.\n- Treated the Bureau's explosibility index as a historical empirical ranking and did not map it to modern KSt classes.\n'''
text += limitations
out.write_text(text)

required=['Original front matter','Abstract','Introduction','Sample Description and Reporting Procedures','Discussion of Explosibility Data','Particle Diameter','Ignition Temperature of Dust Layers','Pyrophoricity','Prevention of Ignition and Explosion','Table 1.','Table 2.','Table 3.','Table 4.','TABLE A-1','TABLE A-2','TABLE A-3','FIGURE 1','FIGURE 2','FIGURE 3','Footnotes and cited references','Extraction limitations','Discretionary analytical decisions']
missing=[x for x in required if x.lower() not in text.lower()]
# Check appendix endpoints and figure inventory.
checks={
 'required_missing':missing,
 'appendix_line_1': bool(re.search(r'\|\s*1\s*\|',text)),
 'appendix_line_314': '314' in text,
 'figure_count':len(re.findall(r'^### Figure [123]$',text,flags=re.M)),
 'main_table1_rows':len(t1_rows),
 'table4_rows':len(t4_rows),
 'footnotes':len(re.findall(r'^\[\^\d+\]:',text,flags=re.M)),
 'chars':len(text), 'lines':text.count('\n')+1
}
print(json.dumps(checks,indent=2))
assert not missing and checks['figure_count']==3 and checks['main_table1_rows']==55 and checks['table4_rows']==34 and checks['footnotes']==14
print('QA passed; bytes',out.stat().st_size)

{
  "required_missing": [],
  "appendix_line_1": true,
  "appendix_line_314": true,
  "figure_count": 3,
  "main_table1_rows": 55,
  "table4_rows": 34,
  "footnotes": 14,
  "chars": 107852,
  "lines": 1363
}
QA passed; bytes 108635


In [27]:
from e14c.data_storage import upload
import os
artifact_uri=upload('/workspace/Explosibility_of_Metal_Powders_RI_6516_complete.md', description='Complete structured Markdown extraction of Bureau of Mines RI 6516, including full narrative, all tables, figure inventory and digitized trends, references, executive summary, and extraction limitations.', name='Explosibility of Metal Powders - RI 6516 - Complete Markdown', project_id=os.environ.get('PROJECT_ID'))
print(artifact_uri)

Uploaded /workspace/Explosibility_of_Metal_Powders_RI_6516_complete.md to data storage as entry data_entry:explosibility-of-metal-powders-ri-6516-complete-pxs6 under the name 'Explosibility of Metal Powders - RI 6516 - Complete Markdown'.
data_entry:explosibility-of-metal-powders-ri-6516-complete-pxs6
